# SO3LR-SF Python API Examples

This notebook demonstrates the complete Python API for SO3LR-SF, covering:

1. **Single Point Calculation** - Basic energy calculation
2. **Structure Trimming** - Reducing protein size around ligands
3. **Optimization with Constraints** - Structure optimization with selective flexibility
4. **Energy Decomposition Analysis (EDA)** - Component-wise energy breakdown
5. **Explainability** - Per-atom energy contributions with heatmaps
6. **Protein-Ligand Explainability** - Enhanced visualization with interaction fingerprints

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Import SO3LR-SF modules
from src.calculator import So3lrSfCalculator
from src.interaction_energy import protein_ligand_interaction
from src.structure_ops import trim_structure, optimize_structure, create_optimization_constraint
from src.molecule_loader import load_ase_structure

# Setup paths to test data
protein_path = "tests/test_data/2CET_prot.xyz"
protein_path_pdb = protein_path.replace(".xyz", ".pdb")

ligand_path = "tests/test_data/2CET_lig.xyz"

/home/hamza/.cache/pypoetry/virtualenvs/so3lr-sf-nvxEypVy-py3.12/lib/python3.12/site-packages/prolif/datafiles.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/hamza/.cache/pypoetry/virtualenvs/so3lr-sf-nvxEypVy-py3.12/lib/python3.12/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


## 1. Single Point Calculation (End Point Calculation)

Calculate protein-ligand interaction energy for single molecules or all molecules in multi-structure files.

In [2]:
# Initialize calculator
calc = So3lrSfCalculator()

print("=== SINGLE POINT CALCULATION (END POINT CALCULATION) ===")

# Example 1: Single molecule calculation
print(f"Protein: {protein_path}")
print(f"Ligand: {ligand_path}")

interaction_energy = protein_ligand_interaction(
    protein_path=protein_path,
    ligand_path=ligand_path, 
    calc=calc,
    verbose=True
)

print(f"Interaction energy: {interaction_energy:.6f} eV")
print(f"Binding energy: {interaction_energy * 23.06:.2f} kcal/mol")

MLFF:root:Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 12:10:07 - INFO - Protein: tests/test_data/2CET_prot.xyz
10-24 12:10:07 - INFO - Ligand: tests/test_data/2CET_lig.xyz
10-24 12:10:07 - DEBUG - Using calculator with model: /home/hamza/github/so3lr-sf/so3lr/params
10-24 12:10:07 - DEBUG - Reading protein structure...
10-24 12:10:07 - DEBUG - Protein loaded: 281 atoms
10-24 12:10:07 - DEBUG - Reading ligand structure...
10-24 12:10:07 - DEBUG - Ligand loaded: 43 atoms
10-24 12:10:07 - DEBUG - Creating complex by concatenating protein and ligand
10-24 12:10:07 - DEBUG - Complex created: 324 total atoms
10-24 12:10:07 - INFO - Calculating non-interacting energy...
10-24 12:10:07 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 12:10:07 - MLFF - Running a mod

=== SINGLE POINT CALCULATION (END POINT CALCULATION) ===
Protein: tests/test_data/2CET_prot.xyz
Ligand: tests/test_data/2CET_lig.xyz


10-24 12:10:10 - INFO - Calculating complex energy...
10-24 12:10:10 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 12:10:10 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 12:10:12 - DEBUG - Complex energy: -576.397583 eV
10-24 12:10:12 - INFO - Interaction energy calculated: -7.156372 eV
10-24 12:10:12 - INFO - Binding energy: -165.0 kcal/mol


Interaction energy: -7.156372 eV
Binding energy: -165.03 kcal/mol


## 2. Structure Trimming

#### a. Reduce protein size by keeping only **atoms (using XYZ format)** within a defined radius

In [3]:
print("\n=== STRUCTURE TRIMMING ===")

# Load original protein to see size
original_protein = load_ase_structure(protein_path)[0]
print(f"Original protein atoms: {len(original_protein)}")

# Trim protein around ligand (8 Angstrom radius)
trimmed_protein_path = trim_structure(
    protein_path=protein_path,
    ligand_path=ligand_path,
    radius=4.0,
    output_dir="trimming_example"
)

# Load trimmed protein to see size reduction
trimmed_protein = load_ase_structure(trimmed_protein_path)[0]
print(f"Trimmed protein atoms: {len(trimmed_protein)}")
print(f"Size reduction: {len(original_protein) - len(trimmed_protein)} atoms")
print(f"Trimmed protein saved to: {trimmed_protein_path}")

# Calculate interaction energy with trimmed protein
trimmed_interaction = protein_ligand_interaction(
    protein_path=trimmed_protein_path,
    ligand_path=ligand_path,
    calc=calc
)

print(f"\nInteraction energies:")
print(f"  Original protein: {interaction_energy:.6f} eV")
print(f"  Trimmed protein:  {trimmed_interaction:.6f} eV")
print(f"  Difference:       {abs(interaction_energy - trimmed_interaction):.6f} eV")

10-24 12:10:12 - WARNING - Using atom-based trimming for .XYZ file: 2CET_prot.xyz
10-24 12:10:12 - WARNING - Individual atoms will be trimmed - residues may be incomplete!
10-24 12:10:12 - WARNING - For complete residue trimming, use PDB format input files
10-24 12:10:12 - DEBUG - Atom-based trimming: 117 atoms within 4.0 Å of ligand
10-24 12:10:12 - INFO - Trimmed protein (117 atoms from 281 original atoms) saved to: trimming_example/2CET_prot_trimmed_4.0A_atom.xyz
10-24 12:10:12 - INFO - Trimming method: atom-based
10-24 12:10:12 - INFO - Protein: trimming_example/2CET_prot_trimmed_4.0A_atom.xyz
10-24 12:10:12 - INFO - Ligand: tests/test_data/2CET_lig.xyz
10-24 12:10:12 - DEBUG - Using calculator with model: /home/hamza/github/so3lr-sf/so3lr/params
10-24 12:10:12 - DEBUG - Reading protein structure...
10-24 12:10:12 - DEBUG - Protein loaded: 117 atoms
10-24 12:10:12 - DEBUG - Reading ligand structure...
10-24 12:10:12 - DEBUG - Ligand loaded: 43 atoms
10-24 12:10:12 - DEBUG - Creatin


=== STRUCTURE TRIMMING ===
Original protein atoms: 281
Trimmed protein atoms: 117
Size reduction: 164 atoms
Trimmed protein saved to: trimming_example/2CET_prot_trimmed_4.0A_atom.xyz


10-24 12:10:16 - INFO - Calculating complex energy...
10-24 12:10:16 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 12:10:16 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 12:10:19 - DEBUG - Complex energy: -161.652771 eV
10-24 12:10:19 - INFO - Interaction energy calculated: -5.161606 eV
10-24 12:10:19 - INFO - Binding energy: -119.0 kcal/mol



Interaction energies:
  Original protein: -7.156372 eV
  Trimmed protein:  -5.161606 eV
  Difference:       1.994766 eV


b. Reduce protein size by keeping only **residues (using PDB format)** within a defined radius

In [ ]:
print("\n=== STRUCTURE TRIMMING ===")

# Load original protein to see size
original_protein = load_ase_structure(protein_path_pdb)[0]
print(f"Original protein atoms: {len(original_protein)}")

# Trim protein around ligand (8 Angstrom radius)
trimmed_protein_path = trim_structure(
    protein_path=protein_path_pdb,
    ligand_path=ligand_path,
    radius=8.0,
)

# Load trimmed protein to see size reduction
trimmed_protein = load_ase_structure(trimmed_protein_path)[0]
print(f"Trimmed protein atoms: {len(trimmed_protein)}")
print(f"Size reduction: {len(original_protein) - len(trimmed_protein)} atoms")
print(f"Trimmed protein saved to: {trimmed_protein_path}")

# Calculate interaction energy with trimmed protein
trimmed_interaction = protein_ligand_interaction(
    protein_path=trimmed_protein_path,
    ligand_path=ligand_path,
    calc=calc
)

print(f"\nInteraction energies:")
print(f"  Original protein: {interaction_energy:.6f} eV")
print(f"  Trimmed protein:  {trimmed_interaction:.6f} eV")
print(f"  Difference:       {abs(interaction_energy - trimmed_interaction):.6f} eV")

10-24 12:08:35 - INFO - Using residue-based trimming for PDB file: 2CET_prot.pdb
10-24 12:08:35 - INFO - Complete residues will be included if any atom is within the cutoff radius
10-24 12:08:35 - INFO - The attribute(s) types have already been read from the topology file. The guesser will only guess empty values for this attribute, if any exists. To overwrite it by completely guessed values, you can pass the attribute to the force_guess parameter instead of the to_guess one
10-24 12:08:35 - INFO - There is no empty types values. Guesser did not guess any new values for types attribute
10-24 12:08:35 - INFO - attribute masses has been guessed successfully.
/home/hamza/.cache/pypoetry/virtualenvs/so3lr-sf-nvxEypVy-py3.12/lib/python3.12/site-packages/MDAnalysis/topology/guessers.py:184: DeprecationWarning: `guess_atom_element` is deprecated!
`guess_atom_element` will be removed in release 3.0.0.
MDAnalysis.topology.guessers is deprecated in favour of the new Guessers API. See MDAnalysis.


=== STRUCTURE TRIMMING ===
Original protein atoms: 324
Trimmed protein atoms: 324
Size reduction: 0 atoms
Trimmed protein saved to: trimming_example/2CET_prot_trimmed_8.0A_residue.xyz


10-24 12:08:38 - INFO - Calculating complex energy...
10-24 12:08:38 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 12:08:38 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 12:08:40 - DEBUG - Complex energy: nan eV
10-24 12:08:40 - INFO - Interaction energy calculated: nan eV
10-24 12:08:40 - INFO - Binding energy: nan kcal/mol



Interaction energies:
  Original protein: -7.156372 eV
  Trimmed protein:  nan eV
  Difference:       nan eV


## 3. Structure Optimization with Constraints

Optimize molecular structures with selective flexibility based on distance from ligand.

In [11]:
print("\n=== STRUCTURE OPTIMIZATION WITH CONSTRAINTS ===")

# Load structures
protein_atoms = load_ase_structure(protein_path)[0]
ligand_atoms = load_ase_structure(ligand_path)[0]

# Create complex
complex_atoms = protein_atoms + ligand_atoms
n_protein_atoms = len(protein_atoms)

# Create constraint (only atoms within 5Å of ligand will be flexible)
constraint = create_optimization_constraint(
    complex_atoms=complex_atoms,
    n_protein_atoms=n_protein_atoms,
    opt_radius=5.0,
    protein_path=protein_path
)

# Optimize structure with constraints
optimized_path, opt_info = optimize_structure(
    atoms=complex_atoms,
    calc=calc,
    optimizer='FIRE',
    fmax=0.05,
    steps=50,
    output_path=Path("tests/test_data/2CET_cpx_optimized.xyz"),
    opt_radius=2.0,
    n_protein_atoms=n_protein_atoms,
    protein_path=protein_path
)

# Show constraint details
constraint_info = opt_info['constraint_info']
constraint_info

10-24 11:38:57 - WARNING - Using atom-level constraints for .XYZ file: 2CET_prot.xyz
10-24 11:38:57 - WARNING - Individual atoms will be constrained - residues may be split!
10-24 11:38:57 - WARNING - For complete residue constraints, use PDB format input files
10-24 11:38:57 - DEBUG - Atom-based constraints: 180 flexible, 101 fixed
10-24 11:38:57 - INFO - Optimization constraints (atom-based):
10-24 11:38:57 - INFO -   Flexible protein atoms: 180/281
10-24 11:38:57 - INFO -   Fixed protein atoms: 101
10-24 11:38:57 - INFO -   Ligand atoms (always flexible): 43
10-24 11:38:57 - WARNING - Using atom-level constraints for .XYZ file: 2CET_prot.xyz
10-24 11:38:57 - WARNING - Individual atoms will be constrained - residues may be split!
10-24 11:38:57 - WARNING - For complete residue constraints, use PDB format input files
10-24 11:38:57 - DEBUG - Atom-based constraints: 11 flexible, 270 fixed
10-24 11:38:57 - INFO - Optimization constraints (atom-based):
10-24 11:38:57 - INFO -   Flexible 


=== STRUCTURE OPTIMIZATION WITH CONSTRAINTS ===
Optimization converged in 50 steps
Energy change: -0.736267 eV
Structure saved to: tests/test_data/2CET_cpx_optimized.xyz


{'constraint_applied': True,
 'constraint_type': 'FixAtoms',
 'optimization_radius': 2.0,
 'total_protein_atoms': 281,
 'total_ligand_atoms': 43,
 'flexible_protein_atoms': 11,
 'fixed_protein_atoms': 270,
 'ligand_atoms_always_flexible': 43,
 'constraint_details': '11/281 protein atoms flexible within 2.0Å of ligand'}

## 4. Energy Decomposition Analysis (EDA)

Break down interaction energy into individual components (MLFF, Electrostatics, Dispersion, etc.).

In [7]:
print("\n=== ENERGY DECOMPOSITION ANALYSIS (EDA) ===")

# Initialize calculator with per-atom components enabled
eda_calc = So3lrSfCalculator(output_per_atom_energy_components=True)

# Calculate interaction energy with EDA
interaction_energy, analysis = protein_ligand_interaction(
    protein_path=protein_path,
    ligand_path=ligand_path,
    calc=eda_calc,
    eda=True,
    verbose=True
)

print(f"\nTotal interaction energy: {interaction_energy:.6f} eV")
print(f"\n=== ENERGY COMPONENT BREAKDOWN ===")

# Display interaction energy components
if 'interaction_energy_components' in analysis:
    components = analysis['interaction_energy_components']
    print("\nInteraction energy components:")
    total_check = 0.0
    for component, energy in components.items():
        print(f"  {component:20s}: {energy:10.6f} eV ({energy*23.06:8.2f} kcal/mol)")
        total_check += energy
    
    print(f"  {'Total (sum)':20s}: {total_check:10.6f} eV ({total_check*23.06:8.2f} kcal/mol)")
    print(f"  {'Total (direct)':20s}: {interaction_energy:10.6f} eV ({interaction_energy*23.06:8.2f} kcal/mol)")

# Show complex, protein, and ligand component totals
for system_type in ['complex', 'protein', 'ligand']:
    key = f'{system_type}_energy_components'
    if key in analysis:
        print(f"\n{system_type.capitalize()} energy components:")
        for component, energy in analysis[key].items():
            print(f"  {component:20s}: {energy:10.6f} eV")

10-24 11:37:19 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 11:37:19 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 11:37:19 - INFO - Protein: tests/test_data/2CET_prot.xyz
10-24 11:37:19 - INFO - Ligand: tests/test_data/2CET_lig.xyz
10-24 11:37:19 - DEBUG - Using calculator with model: /home/hamza/github/so3lr-sf/so3lr/params
10-24 11:37:19 - DEBUG - Reading protein structure...
10-24 11:37:19 - DEBUG - Protein loaded: 281 atoms
10-24 11:37:19 - DEBUG - Reading ligand structure...
10-24 11:37:19 - DEBUG - Ligand loaded: 43 atoms
10-24 11:37:19 - DEBUG - Creating complex by concatenating protein and ligand
10-24 11:37:19 - DEBUG - Complex created: 324 total atoms
10-24 11:37:19 - INFO - Calculating non-interacting energy...
10-24 11:37:19 - WARNIN


=== ENERGY DECOMPOSITION ANALYSIS (EDA) ===


10-24 11:37:25 - DEBUG - Protein per-atom components extracted
10-24 11:37:25 - DEBUG - Ligand per-atom components extracted
10-24 11:37:25 - INFO - Calculating complex energy...
10-24 11:37:25 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 11:37:25 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 11:37:29 - DEBUG - Complex energy: -576.397583 eV
10-24 11:37:29 - DEBUG - Complex per-atom components extracted
10-24 11:37:29 - INFO - Interaction energy calculated: -7.156372 eV
10-24 11:37:29 - INFO - Binding energy: -165.0 kcal/mol
10-24 11:37:29 - INFO - Computing Energy Decomposition Analysis...
10-24 11:37:29 - DEBUG - Component 'dispersion_energy': protein=-5.068500, ligand=-0.641374, complex=-7.299526, interaction=-1.589653 eV
10-24 11:37:29 - DEBU


Total interaction energy: -7.156372 eV

=== ENERGY COMPONENT BREAKDOWN ===

Interaction energy components:
  dispersion_energy   :  -1.589653 eV (  -36.66 kcal/mol)
  mlff_atomic_energy  :  -5.528023 eV ( -127.48 kcal/mol)
  zbl_repulsion       :  -0.000316 eV (   -0.01 kcal/mol)
  electrostatic_energy:  -0.038422 eV (   -0.89 kcal/mol)
  Total (sum)         :  -7.156414 eV ( -165.03 kcal/mol)
  Total (direct)      :  -7.156372 eV ( -165.03 kcal/mol)

Complex energy components:
  dispersion_energy   :  -7.299526 eV
  mlff_atomic_energy  : -585.350464 eV
  zbl_repulsion       :  29.393694 eV
  electrostatic_energy: -13.141303 eV

Protein energy components:
  dispersion_energy   :  -5.068500 eV
  mlff_atomic_energy  : -504.668488 eV
  zbl_repulsion       :  25.296520 eV
  electrostatic_energy: -11.231137 eV

Ligand energy components:
  dispersion_energy   :  -0.641374 eV
  mlff_atomic_energy  : -75.153954 eV
  zbl_repulsion       :   4.097490 eV
  electrostatic_energy:  -1.871744 eV


## 5. Ligand Explainability

Generate per-atom energy contributions and molecular heatmaps showing which atoms contribute most to binding.

In [ ]:
print("\n=== PROTEIN-LIGAND EXPLAINABILITY ===")

# Enhanced explainability requires preparing protein data for ProLIF analysis
# Let's create the required protein prolif data manually

try:
    # Import required modules for enhanced explainability
    import prolif as plf
    import MDAnalysis as mda
    from src.molecule_loader import load_molecule_to_prolif, create_residue_atom_mapping, prepare_mda_universe
    
    print("ProLIF and MDAnalysis available - enabling enhanced explainability")
    
    # Prepare protein for ProLIF analysis
    # Load protein using MDAnalysis
    try:
        protein_universe = prepare_mda_universe(protein_path)
        protein_prolif = plf.Molecule.from_mda(protein_universe, selection="protein")
        
        # Create residue mapping
        residue_mapping = create_residue_atom_mapping(protein_universe)
        
        # Create preloaded protein prolif tuple
        preloaded_protein_prolif = (protein_prolif, residue_mapping)
        
        print(f"Protein prepared for enhanced explainability")
        print(f"Number of residues: {len(residue_mapping)}")
        print(f"Residues found: {list(residue_mapping.keys())}")
        
        # Calculate interaction energy with enhanced explainability
        interaction_energy, analysis = protein_ligand_interaction(
            protein_path=protein_path,
            ligand_path=ligand_path,
            calc=eda_calc,
            explainability=True,
            heatmap_output="explainability_example/protein_ligand_heatmap.png",
            preloaded_protein_prolif=preloaded_protein_prolif,
            verbose=True
        )
        
        print(f"\nEnhanced interaction energy: {interaction_energy:.6f} eV")
        
        if 'heatmap_path' in analysis and analysis['heatmap_path']:
            print(f"Enhanced heatmap saved to: {analysis['heatmap_path']}")
            
            # Check if file was actually created
            heatmap_path = Path(analysis['heatmap_path'])
            if heatmap_path.exists():
                print(f"✓ Enhanced heatmap file exists ({heatmap_path.stat().st_size} bytes)")
            else:
                print(f"✗ Enhanced heatmap file was not created")
        else:
            print("✗ No heatmap path in analysis results")
        
        # The enhanced visualization shows:
        print("\nEnhanced visualization features:")
        print("  ✓ Ligand atoms colored by energy contribution")
        print("  ✓ Interacting protein residues shown as connected nodes")
        print("  ✓ Bonds colored by interaction type")
        print("  ✓ Combined ligand + protein residue energy contributions")
        print("  ✓ Interaction fingerprint legend")
        
        # Show interaction analysis if available
        if 'interaction_fingerprint' in analysis:
            fp_data = analysis['interaction_fingerprint']
            print(f"\nProtein-ligand interactions detected:")
            for interaction in fp_data:
                print(f"  {interaction['protein_residue']} - {interaction['interaction_type']}")
        
    except Exception as prolif_error:
        print(f"Enhanced explainability setup failed: {prolif_error}")
        print("This might be due to:")
        print("  - Protein structure format issues")
        print("  - Missing ProLIF dependencies")
        print("  - Small test system limitations")
        raise prolif_error
        
except ImportError as import_error:
    print(f"Enhanced explainability requires ProLIF and MDAnalysis: {import_error}")
    print("Falling back to basic explainability...")
    
    # Fallback to basic explainability
    interaction_energy, analysis = protein_ligand_interaction(
        protein_path=protein_path,
        ligand_path=ligand_path,
        calc=eda_calc,
        explainability=True,
        heatmap_output="explainability_example/fallback_heatmap.png",
        verbose=True
    )
    
    print(f"\nFallback interaction energy: {interaction_energy:.6f} eV")
    
    if 'heatmap_path' in analysis and analysis['heatmap_path']:
        print(f"Fallback heatmap saved to: {analysis['heatmap_path']}")
        
        # Check if file was actually created
        heatmap_path = Path(analysis['heatmap_path'])
        if heatmap_path.exists():
            print(f"✓ Fallback heatmap file exists ({heatmap_path.stat().st_size} bytes)")
        else:
            print(f"✗ Fallback heatmap file was not created")
    else:
        print("✗ No heatmap path in analysis results")

except Exception as e:
    print(f"Explainability failed: {e}")
    print("\nDiagnostic information:")
    print(f"  Protein path exists: {Path(protein_path).exists()}")
    print(f"  Ligand path exists: {Path(ligand_path).exists()}")
    print(f"  Output directory: explainability_example/")
    
    # Create output directory if it doesn't exist
    Path("explainability_example").mkdir(exist_ok=True)
    print(f"  Output directory created/exists: {Path('explainability_example').exists()}")

# Additional check for any generated files
print(f"\nChecking for generated files in explainability_example/:")
exp_dir = Path("explainability_example")
if exp_dir.exists():
    for file in exp_dir.iterdir():
        print(f"  Found: {file.name} ({file.stat().st_size} bytes)")
else:
    print("  Directory does not exist")

## 6. Protein-Ligand Explainability

Enhanced explainability with protein-ligand interaction fingerprints, showing both ligand contributions and interacting protein residues.

In [9]:
print("\n=== PROTEIN-LIGAND EXPLAINABILITY ===")

# First, we need to prepare protein data for protein-ligand explainability
from src.molecule_loader import load_molecule_to_prolif

# Load and prepare protein for ProLIF analysis
## !!!!!!!!! Make sure to use PDB format for protein-ligand explainability

protein_path_pdb = protein_path.replace(".xyz", ".pdb")
preloaded_protein_prolif = load_molecule_to_prolif(protein_path_pdb)

# Calculate interaction energy with enhanced explainability
interaction_energy, analysis = protein_ligand_interaction(
    protein_path=protein_path_pdb,
    ligand_path=ligand_path,
    calc=eda_calc,
    explainability=True,
    heatmap_output="tests/test_data/protein_ligand_heatmap.png",
    preloaded_protein_prolif=preloaded_protein_prolif,
    verbose=True
)

10-24 11:37:37 - INFO - Protein: tests/test_data/2CET_prot.pdb
10-24 11:37:37 - INFO - Ligand: tests/test_data/2CET_lig.xyz
10-24 11:37:37 - DEBUG - Using calculator with model: /home/hamza/github/so3lr-sf/so3lr/params
10-24 11:37:37 - DEBUG - Reading protein structure...
10-24 11:37:37 - DEBUG - Protein loaded: 324 atoms
10-24 11:37:37 - DEBUG - Reading ligand structure...
10-24 11:37:37 - DEBUG - Ligand loaded: 43 atoms
10-24 11:37:37 - DEBUG - Creating complex by concatenating protein and ligand
10-24 11:37:37 - DEBUG - Complex created: 367 total atoms
10-24 11:37:37 - INFO - Calculating non-interacting energy...
10-24 11:37:37 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 11:37:37 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.



=== PROTEIN-LIGAND EXPLAINABILITY ===


10-24 11:37:43 - DEBUG - Protein per-atom components extracted
10-24 11:37:43 - DEBUG - Ligand per-atom components extracted
10-24 11:37:43 - INFO - Calculating complex energy...
10-24 11:37:43 - WARNING - `create_from_ckpt_dir` is deprecated and replaced by `create_from_workdir`, please use this method in the future. For now this calls `create_from_workdir` but will raise an error in the future.
10-24 11:37:43 - MLFF - Running a model with long-range corrections. The local cutoff is 4.5 Ang and the long-range cutoff is 12.0.
10-24 11:37:48 - DEBUG - Complex energy: nan eV
10-24 11:37:48 - DEBUG - Complex per-atom components extracted
10-24 11:37:48 - INFO - Interaction energy calculated: nan eV
10-24 11:37:48 - INFO - Binding energy: nan kcal/mol
10-24 11:37:48 - INFO - Starting explainability analysis...
10-24 11:37:48 - DEBUG - Atom counts - Protein: 324, Ligand: 43
10-24 11:37:48 - DEBUG - Computing per-atom energy differences for ligand atoms...
10-24 11:37:48 - INFO - Generating 

  0%|          | 0/1 [00:00<?, ?it/s]

10-24 11:37:50 - WARNING - Could not generate heatmap: 'Residue' object is not subscriptable
10-24 11:37:50 - INFO - Explainability analysis complete


## Summary and Advanced Usage

### Key API Functions:

1. **`So3lrSfCalculator`** - Main calculator class
   - `output_per_atom_energy_components=True` for EDA/explainability
   - Auto-detects model path if not specified

2. **`protein_ligand_interaction`** - Main calculation function
   - `explainability=True` for per-atom heatmaps
   - `eda=True` for energy component breakdown
   - `preloaded_protein_prolif` for enhanced visualization

3. **`trim_structure`** - Reduce protein size
   - Residue-based trimming for PDB files
   - Atom-based trimming for XYZ/SDF files

4. **`optimize_structure`** - Structure optimization
   - `opt_radius` for selective flexibility
   - FIRE or LBFGS optimizers
   - Automatic constraint generation

### Workflow Combinations:

```python
# Complete workflow with all features
calc = So3lrSfCalculator(output_per_atom_energy_components=True)

# 1. Trim protein
trimmed_protein = trim_structure(protein_path, ligand_path, radius=10.0)

# 2. Optimize with constraints
protein_atoms = load_ase_structure(trimmed_protein)[0]
ligand_atoms = load_ase_structure(ligand_path)[0]
complex_atoms = protein_atoms + ligand_atoms

opt_path, _ = optimize_structure(
    complex_atoms, calc, opt_radius=8.0, 
    n_protein_atoms=len(protein_atoms)
)

# 3. Full analysis
interaction_energy, analysis = protein_ligand_interaction(
    trimmed_protein, ligand_path, calc,
    complex_path=opt_path,
    explainability=True,
    eda=True,
    heatmap_output="complete_analysis.png"
)
```